In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, RandomRotation
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

IMAGE_SHAPE = (96, 96)  # Smaller than 224x224 — faster, better for upscaled 32px images
BATCH_SIZE = 64

# ── Load CIFAR-10 ────────────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.cifar10.load_data()

# ── Data augmentation (applied to raw pixels, before preprocess_input) ──
data_augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.RandomBrightness(0.2),
])


def preprocess_train(image, label):
    image = tf.cast(image, tf.float32)                 # keep in [0, 255] for augmentation
    image = data_augment(image, training=True)         # augment on raw pixels
    image = tf.image.resize(image, IMAGE_SHAPE)        # then resize
    image = preprocess_input(image)                    # then normalize
    return image, label

def preprocess_val(image, label):
    image = tf.cast(image, tf.float32)
    image = tf.image.resize(image, IMAGE_SHAPE)
    image = preprocess_input(image)
    return image, label

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE) \
    .shuffle(10000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)) \
    .map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

Data Augmentation

In [2]:

# ── Model ────────────────────────────────────────────────────────
base_model = ResNet50(
    input_shape=IMAGE_SHAPE + (3,),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

classifier = tf.keras.Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.2),
    Dense(10, activation="softmax")
])

classifier.summary()
# ── Phase 1: Train head only ─────────────────────────────────────
# Use val dataset (no augmentation) to let head converge faster
train_ds_noaug = tf.data.Dataset.from_tensor_slices((X_train, y_train)) \
    .map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE) \
    .shuffle(10000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

early_stopping_phase1 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_head = classifier.fit(
    train_ds_noaug,       # no augmentation — let head converge cleanly
    epochs=15,
    validation_data=test_ds,
    callbacks=[early_stopping_phase1]
)

# ── Phase 2: Fine-tune with augmentation + more patience ─────────
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

early_stopping_phase2 = EarlyStopping(
    monitor="val_loss",
    patience=6,                    # slow lr needs more patience
    restore_best_weights=True
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,                    # halve lr if stuck for 3 epochs
    min_lr=1e-7
)

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_finetune = classifier.fit(
    train_ds,                      # augmented dataset for fine-tuning
    epochs=40,                     # give it room — early stopping will cut it short
    validation_data=test_ds,
    callbacks=[early_stopping_phase2, lr_scheduler]
)

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 3, 3, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,851,274 (90.99 MB)

 Trainable params: 263,562 (1.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

Epoch 1/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 65s 62ms/step - accuracy: 0.8046 - loss: 0.5821 - val_accuracy: 0.8480 - val_loss: 0.4297
Epoch 2/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 40s 47ms/step - accuracy: 0.8558 - loss: 0.4171 - val_accuracy: 0.8535 - val_loss: 0.4178
Epoch 3/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 45s 53ms/step - accuracy: 0.8757 - loss: 0.3594 - val_accuracy: 0.8589 - val_loss: 0.4173
Epoch 4/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 45s 52ms/step - accuracy: 0.8863 - loss: 0.3192 - val_accuracy: 0.8654 - val_loss: 0.4045
Epoch 5/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 40s 47ms/step - accuracy: 0.8986 - loss: 0.2865 - val_accuracy: 0.8671 - val_loss: 0.4034
Epoch 6/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 40s 46ms/step - accuracy: 0.9073 - loss: 0.2596 - val_accuracy: 0.8636 - val_loss: 0.4163
Epoch 7/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 40s 46ms/step - accuracy: 0.9162 - loss: 0.2334 - val_accuracy: 0.8671 - val_loss: 0.4279
Epoch 8/15
782/782 ━━━━━━━━━━━━━━━━━━━━ 41s 47ms/step - accuracy: 0.9204 - loss: 0.2148 - 

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# ── Generate predictions on the test set ─────────────────────────
print("Running predictions on test set...")
y_pred_probs = classifier.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

# Reconstruct true labels in the same order as the batched test_ds
y_true = np.concatenate([y.numpy().flatten() for _, y in test_ds])

# ── Confusion Matrix ─────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
    linewidths=0.4,
    linecolor='#e2e8f0',
    ax=ax,
    cbar_kws={'shrink': 0.8}
)

ax.set_xlabel('Predicted Label', fontsize=12, labelpad=12)
ax.set_ylabel('True Label', fontsize=12, labelpad=12)
ax.set_title(
    'Confusion Matrix — ResNet50 + Augmentation + LR Scheduler',
    fontsize=14, fontweight='bold', pad=16
)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.show()

# ── Per-class precision / recall / F1 ────────────────────────────
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ── Highlight the most-confused pairs ────────────────────────────
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
top_k = 5
flat_idx = np.argsort(cm_no_diag.ravel())[::-1][:top_k]
print(f"\nTop-{top_k} most confused pairs (true → predicted):")
for idx in flat_idx:
    r, c = divmod(idx, len(class_names))
    print(f"  {class_names[r]:12s} → {class_names[c]:12s}  ({cm[r, c]} misclassified)")